In [1]:
#phase - 0
!pip install pandas numpy scikit-learn matplotlib seaborn flask ultralytics opencv-python streamlit

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import cv2
import flask
import streamlit
from ultralytics import YOLO

In [3]:
df = pd.read_csv('dataset_traffic_accident_prediction1.csv')  
print(df.shape) 
print(df.columns.tolist()) 
df.head() 

(840, 14)
['Weather', 'Road_Type', 'Time_of_Day', 'Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles', 'Driver_Alcohol', 'Accident_Severity', 'Road_Condition', 'Vehicle_Type', 'Driver_Age', 'Driver_Experience', 'Road_Light_Condition', 'Accident']


,Weather,Road_Type,Time_of_Day,Traffic_Density,Speed_Limit,Number_of_Vehicles,Driver_Alcohol,Accident_Severity,Road_Condition,Vehicle_Type,Driver_Age,Driver_Experience,Road_Light_Condition,Accident
0,Rainy,City Road,Morning,1.0,100.0,5.0,0.0,NaN,Wet,Car,51.0,48.0,Artificial Light,0.0
1,Clear,Rural Road,Night,NaN,120.0,3.0,0.0,Moderate,Wet,Truck,49.0,43.0,Artificial Light,0.0
2,Rainy,Highway,Evening,1.0,60.0,4.0,0.0,Low,Icy,Car,54.0,52.0,Artificial Light,0.0
3,Clear,City Road,Afternoon,2.0,60.0,3.0,0.0,Low,Under Construction,Bus,34.0,31.0,Daylight,0.0
4,Rainy,Highway,Morning,1.0,195.0,11.0,0.0,Low,Dry,Car,62.0,55.0,Artificial Light,1.0


In [4]:
print(df['Accident'].value_counts()) 
print(df['Accident'].value_counts(normalize=True) * 100) 

Accident
0.0    559
1.0    239
Name: count, dtype: int64
Accident
0.0    70.050125
1.0    29.949875
Name: proportion, dtype: float64


In [5]:
print(df.isnull().sum()) 

Weather                 42
Road_Type               42
Time_of_Day             42
Traffic_Density         42
Speed_Limit             42
Number_of_Vehicles      42
Driver_Alcohol          42
Accident_Severity       42
Road_Condition          42
Vehicle_Type            42
Driver_Age              42
Driver_Experience       42
Road_Light_Condition    42
Accident                42
dtype: int64


In [6]:
print(df[df.isnull().all(axis=1)].shape) 

(0, 14)


In [7]:
print(df[df.isnull().any(axis=1)].shape)

(435, 14)


In [8]:
df = df.dropna(subset=['Accident'])
print(df.shape)

(798, 14)


In [9]:
print(df.isnull().sum())

Weather                 40
Road_Type               40
Time_of_Day             38
Traffic_Density         40
Speed_Limit             41
Number_of_Vehicles      38
Driver_Alcohol          40
Accident_Severity       42
Road_Condition          41
Vehicle_Type            39
Driver_Age              40
Driver_Experience       41
Road_Light_Condition    40
Accident                 0
dtype: int64


In [10]:
numeric_cols = ['Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles', 'Driver_Age', 'Driver_Experience']
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())


categorical_cols = ['Weather', 'Road_Type', 'Time_of_Day', 'Driver_Alcohol', 'Road_Condition', 'Vehicle_Type', 'Road_Light_Condition']
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(df.isnull().sum())

Weather                  0
Road_Type                0
Time_of_Day              0
Traffic_Density          0
Speed_Limit              0
Number_of_Vehicles       0
Driver_Alcohol           0
Accident_Severity       42
Road_Condition           0
Vehicle_Type             0
Driver_Age               0
Driver_Experience        0
Road_Light_Condition     0
Accident                 0
dtype: int64


In [11]:
categorical_cols = ['Weather', 'Road_Type', 'Time_of_Day', 'Driver_Alcohol', 'Road_Condition', 'Vehicle_Type', 'Road_Light_Condition']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print(df_encoded.shape)
df_encoded.head()

(798, 26)


,Traffic_Density,Speed_Limit,Number_of_Vehicles,Accident_Severity,Driver_Age,Driver_Experience,Accident,Weather_Foggy,Weather_Rainy,Weather_Snowy,...,Time_of_Day_Night,Driver_Alcohol_1.0,Road_Condition_Icy,Road_Condition_Under Construction,Road_Condition_Wet,Vehicle_Type_Car,Vehicle_Type_Motorcycle,Vehicle_Type_Truck,Road_Light_Condition_Daylight,Road_Light_Condition_No Light
0,1.0,100.0,5.0,NaN,51.0,48.0,0.0,False,True,False,...,False,False,False,False,True,True,False,False,False,False
1,1.0,120.0,3.0,Moderate,49.0,43.0,0.0,False,False,False,...,True,False,False,False,True,False,False,True,False,False
2,1.0,60.0,4.0,Low,54.0,52.0,0.0,False,True,False,...,False,False,True,False,False,True,False,False,False,False
3,2.0,60.0,3.0,Low,34.0,31.0,0.0,False,False,False,...,False,False,False,True,False,False,False,False,True,False
4,1.0,195.0,11.0,Low,62.0,55.0,1.0,False,True,False,...,False,False,False,False,False,True,False,False,False,False


In [12]:
##phase - 1
df_model = df_encoded.drop(columns=['Accident_Severity'])
print(df_model.shape)
df_model.columns.tolist()

(798, 25)


['Traffic_Density',
 'Speed_Limit',
 'Number_of_Vehicles',
 'Driver_Age',
 'Driver_Experience',
 'Accident',
 'Weather_Foggy',
 'Weather_Rainy',
 'Weather_Snowy',
 'Weather_Stormy',
 'Road_Type_Highway',
 'Road_Type_Mountain Road',
 'Road_Type_Rural Road',
 'Time_of_Day_Evening',
 'Time_of_Day_Morning',
 'Time_of_Day_Night',
 'Driver_Alcohol_1.0',
 'Road_Condition_Icy',
 'Road_Condition_Under Construction',
 'Road_Condition_Wet',
 'Vehicle_Type_Car',
 'Vehicle_Type_Motorcycle',
 'Vehicle_Type_Truck',
 'Road_Light_Condition_Daylight',
 'Road_Light_Condition_No Light']

In [13]:
X = df_model.drop(columns=['Accident'])
y = df_model['Accident']

print(X.shape)
print(y.shape)

(798, 24)
(798,)


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(638, 24) (160, 24)
Accident
0.0    447
1.0    191
Name: count, dtype: int64
Accident
0.0    112
1.0     48
Name: count, dtype: int64


In [15]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [16]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.73125

Classification Report:
               precision    recall  f1-score   support

         0.0       0.72      1.00      0.84       112
         1.0       1.00      0.10      0.19        48

    accuracy                           0.73       160
   macro avg       0.86      0.55      0.51       160
weighted avg       0.81      0.73      0.64       160


Confusion Matrix:
 [[112   0]
 [ 43   5]]


In [17]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.68125

Classification Report:
               precision    recall  f1-score   support

         0.0       0.73      0.88      0.79       112
         1.0       0.44      0.23      0.30        48

    accuracy                           0.68       160
   macro avg       0.58      0.55      0.55       160
weighted avg       0.64      0.68      0.65       160


Confusion Matrix:
 [[98 14]
 [37 11]]


In [18]:
y_probs = model.predict_proba(X_test)[:, 1]  

threshold = 0.3
y_pred_adjusted = (y_probs >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred_adjusted))
print("\nClassification Report:\n", classification_report(y_test, y_pred_adjusted))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_adjusted))

Accuracy: 0.43125

Classification Report:
               precision    recall  f1-score   support

         0.0       0.76      0.28      0.41       112
         1.0       0.32      0.79      0.46        48

    accuracy                           0.43       160
   macro avg       0.54      0.53      0.43       160
weighted avg       0.63      0.43      0.42       160


Confusion Matrix:
 [[31 81]
 [10 38]]


In [19]:
threshold = 0.4
y_pred_adjusted = (y_probs >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred_adjusted))
print("\nClassification Report:\n", classification_report(y_test, y_pred_adjusted))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_adjusted))

Accuracy: 0.6

Classification Report:
               precision    recall  f1-score   support

         0.0       0.77      0.62      0.68       112
         1.0       0.39      0.56      0.46        48

    accuracy                           0.60       160
   macro avg       0.58      0.59      0.57       160
weighted avg       0.65      0.60      0.62       160


Confusion Matrix:
 [[69 43]
 [21 27]]


In [20]:
import joblib

joblib.dump(model, 'accident_risk_model.pkl')
print("Model saved successfully")

Model saved successfully


In [21]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\ujwal\Downloads\AI_ASS_Project
['accident_risk_model.pkl', 'dataset_traffic_accident_prediction1.csv', 'Phase_0 (Setup).ipynb', 'Phase_0-8.ipynb', 'runs', 'traffic_video.mp4', 'yolov8n.pt']


In [22]:
import os
print(os.listdir())

['accident_risk_model.pkl', 'dataset_traffic_accident_prediction1.csv', 'Phase_0 (Setup).ipynb', 'Phase_0-8.ipynb', 'runs', 'traffic_video.mp4', 'yolov8n.pt']


In [23]:
from ultralytics import YOLO
model_yolo = YOLO('yolov8n.pt')

In [24]:
results = model_yolo('traffic_video.mp4', save=True)
print("Detection complete")


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1878) c:\Users\ujwal\Downloads\AI_ASS_Project\traffic_video.mp4: 384x640 1 person, 6 cars, 2 traffic lights, 1 fire hydrant, 172.0ms
video 1/1 (frame 2/1878) c:\Users\ujwal\Downloads\AI_ASS_Project\traffic_video.mp4: 384x640 1 person, 5 cars, 3 traffic lights, 2 fire hydrants, 43.7ms
video 1/1 (frame 3/1878) c:\Users\ujwal\Downloads\AI_ASS_Project\traffic_video.mp4: 384x640 1 person, 8 cars, 3 traffic lights, 1 fire hydrant, 42.5ms
video 1

In [25]:
results = model_yolo('traffic_video.mp4', stream=True)

vehicle_classes = ['car', 'truck', 'bus', 'motorcycle']

traffic_counts = []

for r in results:
    count = 0
    for box in r.boxes:
        class_id = int(box.cls[0])
        class_name = model_yolo.names[class_id]
        if class_name in vehicle_classes:
            count += 1
    traffic_counts.append(count)

print(traffic_counts[:20])
print(f"Processed {len(traffic_counts)} frames total")


video 1/1 (frame 1/1878) c:\Users\ujwal\Downloads\AI_ASS_Project\traffic_video.mp4: 384x640 1 person, 6 cars, 2 traffic lights, 1 fire hydrant, 64.8ms
video 1/1 (frame 2/1878) c:\Users\ujwal\Downloads\AI_ASS_Project\traffic_video.mp4: 384x640 1 person, 5 cars, 3 traffic lights, 2 fire hydrants, 49.5ms
video 1/1 (frame 3/1878) c:\Users\ujwal\Downloads\AI_ASS_Project\traffic_video.mp4: 384x640 1 person, 8 cars, 3 traffic lights, 1 fire hydrant, 48.8ms
video 1/1 (frame 4/1878) c:\Users\ujwal\Downloads\AI_ASS_Project\traffic_video.mp4: 384x640 2 persons, 7 cars, 5 traffic lights, 46.2ms
video 1/1 (frame 5/1878) c:\Users\ujwal\Downloads\AI_ASS_Project\traffic_video.mp4: 384x640 2 persons, 5 cars, 5 traffic lights, 1 fire hydrant, 46.2ms
video 1/1 (frame 6/1878) c:\Users\ujwal\Downloads\AI_ASS_Project\traffic_video.mp4: 384x640 1 person, 5 cars, 4 traffic lights, 1 fire hydrant, 46.0ms
video 1/1 (frame 7/1878) c:\Users\ujwal\Downloads\AI_ASS_Project\traffic_video.mp4: 384x640 1 person, 5 ca

In [26]:
#phase 3
import numpy as np

avg_count = np.mean(traffic_counts)
print(f"Average vehicles per frame: {avg_count:.2f}")

if avg_count < 5:
    traffic_density_label = "Low"
elif avg_count < 10:
    traffic_density_label = "Medium"
else:
    traffic_density_label = "High"

print(f"Traffic density: {traffic_density_label}")

Average vehicles per frame: 4.86
Traffic density: Low


In [27]:
import joblib

model = joblib.load("accident_risk_model.pkl")
print(model.feature_names_in_)

['Traffic_Density' 'Speed_Limit' 'Number_of_Vehicles' 'Driver_Age' 'Driver_Experience' 'Weather_Foggy' 'Weather_Rainy' 'Weather_Snowy' 'Weather_Stormy' 'Road_Type_Highway' 'Road_Type_Mountain Road' 'Road_Type_Rural Road' 'Time_of_Day_Evening' 'Time_of_Day_Morning' 'Time_of_Day_Night' 'Driver_Alcohol_1.0'
 'Road_Condition_Icy' 'Road_Condition_Under Construction' 'Road_Condition_Wet' 'Vehicle_Type_Car' 'Vehicle_Type_Motorcycle' 'Vehicle_Type_Truck' 'Road_Light_Condition_Daylight' 'Road_Light_Condition_No Light']


In [28]:
print(df['Traffic_Density'].unique())
print(df['Traffic_Density'].describe())

[          1           2           0]
count    798.000000
mean       1.011278
std        0.770291
min        0.000000
25%        0.000000
50%        1.000000
75%        2.000000
max        2.000000
Name: Traffic_Density, dtype: float64


In [29]:
df_raw = pd.read_csv('dataset_traffic_accident_prediction1.csv')
print(df_raw['Traffic_Density'].unique())

[          1         nan           2           0]


In [31]:
# Phase 4: predict accident risk from video-derived traffic density

# Step 1: map our video label to the model's numeric scale (documented assumption)
density_map = {"Low": 0, "Medium": 1, "High": 2}
density_value = density_map[traffic_density_label]

# Step 2: start with sensible defaults from the training data for everything
# the video can't tell us (weather, road type, driver info, etc.)
input_row = X.mode().iloc[0].copy()  # most common value per column
for col in ['Speed_Limit', 'Number_of_Vehicles', 'Driver_Age', 'Driver_Experience']:
    input_row[col] = X[col].median()  # median is more sensible than mode for these

# Step 3: overwrite Traffic_Density with what we actually derived from the video
input_row['Traffic_Density'] = density_value

# Step 4: match the model's expected column order exactly
input_row = input_row[model.feature_names_in_]

# Step 5: predict
input_df = pd.DataFrame([input_row])
prob = model.predict_proba(input_df)[0][1]
prediction = 1 if prob >= 0.4 else 0  # your chosen threshold from Phase 1

print(f"Accident risk probability: {prob:.2f}")
print(f"Prediction: {'⚠️ ACCIDENT RISK' if prediction == 1 else '✅ LOW RISK'}")

Accident risk probability: 0.29
Prediction: ✅ LOW RISK


In [32]:
import json

defaults = X.mode().iloc[0].to_dict()
for col in ['Speed_Limit', 'Number_of_Vehicles', 'Driver_Age', 'Driver_Experience']:
    defaults[col] = float(X[col].median())

with open('defaults.json', 'w') as f:
    json.dump(defaults, f)
print("saved")

saved
